# Demo: agente de planificacion de ecoturismo con replanificacion

Agencia boutique del sur de Chile: dado un pedido del cliente (actividad, nivel, fecha), el agente:

1. **Recupera** paquetes internos via RAG (embeddings multilingues + ChromaDB).
2. **Verifica** condiciones reales con herramientas: clima (Open-Meteo), estado de senderos, disponibilidad de guias.
3. **Replanifica** si hay conflicto (sendero cerrado, mal clima, guia sin agenda, fuera de temporada) eligiendo una alternativa validada.
4. **Responde** citando cada decision: `[F#]` fragmentos internos, `[T#]` resultados de herramientas.
5. **Registra** cada paso en `logs/trace.jsonl`.

Para que la demo corra sin cuota de Groq y de forma reproducible se usa `ClienteFalso` (determinista). Para probar con LLM real: `agente = AgentePlanificador(llm=ClienteGroq())`.

In [ ]:
from agent.llm_client import ClienteFalso, ClienteGroq
from agent.reasoning_loop import AgentePlanificador
from agent.trace import Trazador
from pathlib import Path
import json

trazador = Trazador(Path("logs") / "trace_demo.ipynb.jsonl")
agente = AgentePlanificador(llm=ClienteFalso(), trazador=trazador)  # swap: llm=ClienteGroq()

## Caso 1: pedido sin conflicto

Kayak principiante en Chiloe el 2026-12-08: sendero abierto, dentro de temporada y guia disponible. Respuesta directa, sin replanificacion.

In [ ]:
r = agente.planificar("kayak suave para principiantes en Chiloe", fecha="2026-12-08")
print(r.texto)
print()
print("Paquete:", r.paquete_id, "| Replanificado:", r.replanificado, "| Conflictos:", r.detalle_conflicto)
print("Fuentes:", r.fuentes_citadas)

## Caso 2: conflicto -> replanificacion automatica

Observacion de fauna en Los Lagos el 2026-12-05: el mejor paquete (PAQ-005, ballenas) esta fuera de temporada y su guia no tiene agenda. El replanificador busca alternativas, las valida (senderos + guia + clima) y recomienda PAQ-003 (pinguinera de Punihuil), explicando el cambio.

In [ ]:
r = agente.planificar("observacion de ballenas y fauna marina en Los Lagos", fecha="2026-12-05")
print(r.texto)
print()
print("Original:", r.paquete_original_id, "-> Final:", r.paquete_id, "| Replanificado:", r.replanificado)
print("Conflictos detectados:", r.detalle_conflicto)
print("Fuentes:", r.fuentes_citadas)

## Caso 3: clima adverso (pronostico simulado)

Inyectamos un proveedor de clima *stub* que pronostica tormenta en todo el sur. Ningun paquete escapa al mal clima, asi que el agente responde con **honestidad informativa**: no inventa alternativas.

In [ ]:
def clima_tormenta(region, fecha):
    return {
        "disponible": True, "region": region, "fecha": fecha,
        "temp_max_c": 5.0, "temp_min_c": -4.0, "precipitacion_mm": 25.0,
        "viento_max_kmh": 70.0, "codigo_clima": 80, "fuente": "stub-clima",
    }

agente_tormenta = AgentePlanificador(llm=ClienteFalso(), trazador=trazador, proveedor_clima=clima_tormenta)
r = agente_tormenta.planificar("kayak suave para principiantes en Chiloe", fecha="2026-12-08")
print(r.texto)
print()
print("Paquete:", r.paquete_id, "| Replanificado:", r.replanificado, "| Conflictos:", r.detalle_conflicto)

## Caso 4: replanificacion fallida -> respuesta honesta

Trekking en Torres del Paine el 2026-12-04: el paquete top-1 usa `senda-base-torres` (cerrada) y, al revisar alternativas, ningun guia del roster tiene disponibilidad para esa fecha. No hay alternativa viable y el agente lo declara en vez de forzar una recomendacion.

In [ ]:
r = agente.planificar("trekking exigente en Torres del Paine", fecha="2026-12-04")
print(r.texto)
print()
print("Paquete:", r.paquete_id, "| Replanificado:", r.replanificado, "| Conflictos:", r.detalle_conflicto)

## Trazabilidad completa del loop

Cada paso (consulta, recuperacion, herramientas, replanificacion, respuesta) queda en JSONL. Aqui se muestra el trace de la demo.

In [ ]:
for linea in trazador.archivo.read_text(encoding="utf-8").strip().split("\n"):
    e = json.loads(linea)
    resumen = {k: v for k, v in e.items() if k not in ("paso", "tipo", "hora_utc")}
    print(f"[{e['paso']:02d}] {e['tipo']:<14} {json.dumps(resumen, ensure_ascii=False)[:110]}")

---

**Limitaciones declaradas:** clima real via Open-Meteo (sin key, rango ~16 dias); estado de senderos simulado en `data/external/trail_status.json` (CONAF no tiene API publica); roster de guias interno con fechas ISO exactas.

Ver tambien: `main.py` (CLI) y `tests/` (suite Fase 3 y 4).